In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


INPUT_FILE = "V4 Usage Training.csv"
OUTPUT_FOLDER = Path("forecast_results")

# Future forecast settings:
# Train through June 2026 and forecast July-December 2026.
TRAINING_CUTOFF = "2026-07-01"
FORECAST_MONTHS = 6

TARGET_COLUMN = "Monthly Inventory Issues"
PART_COLUMN = "fpartno"
DATE_COLUMN = "Date"

# Commitment feature
COMMITS_COLUMN = "Avg Commits/Month"

LAGS = [1, 2, 3, 6, 12]

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Version 8 July-December 2026 future forecast "
    "settings loaded."
)


In [ ]:
data = pd.read_csv(INPUT_FILE)

required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    COMMITS_COLUMN,
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

data[DATE_COLUMN] = pd.to_datetime(
    data[DATE_COLUMN],
    errors="raise",
)

# Keep only data from 2023 onward
data = data[
    data[DATE_COLUMN] >= pd.Timestamp("2023-01-01")
].copy()

data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN],
    errors="raise",
)

data[COMMITS_COLUMN] = pd.to_numeric(
    data[COMMITS_COLUMN],
    errors="coerce",
)

# Blank commitments treated as zero
data[COMMITS_COLUMN] = (
    data[COMMITS_COLUMN]
    .fillna(0)
)

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)

if duplicates.any():

    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows were found:\n"
        f"{duplicate_rows.head(20)}"
    )

print(f"Rows loaded: {len(data):,}")
print(f"Unique parts: {data[PART_COLUMN].nunique():,}")
print(f"First date: {data[DATE_COLUMN].min()}")
print(f"Last date: {data[DATE_COLUMN].max()}")
print(f"Average commitments: {data[COMMITS_COLUMN].mean():.2f}")
print(
    f"Rows with commitments: "
    f"{(data[COMMITS_COLUMN] > 0).sum():,}"
)


In [ ]:
def create_training_features(historical_data):

    feature_data = historical_data.copy()

    feature_data["month"] = feature_data[DATE_COLUMN].dt.month
    feature_data["year"] = feature_data[DATE_COLUMN].dt.year
    feature_data["quarter"] = feature_data[DATE_COLUMN].dt.quarter
    feature_data["time_idx"] = np.arange(len(feature_data))

    for lag in LAGS:
        feature_data[f"lag_{lag}"] = (
            feature_data[TARGET_COLUMN].shift(lag)
        )

    prior_usage = feature_data[TARGET_COLUMN].shift(1)

    feature_data["rolling_mean_3"] = prior_usage.rolling(3).mean()
    feature_data["rolling_mean_6"] = prior_usage.rolling(6).mean()
    feature_data["rolling_mean_12"] = prior_usage.rolling(12).mean()

    feature_data["rolling_median_3"] = prior_usage.rolling(3).median()
    feature_data["rolling_median_6"] = prior_usage.rolling(6).median()
    feature_data["rolling_median_12"] = prior_usage.rolling(12).median()

    feature_data["rolling_total_12"] = prior_usage.rolling(12).sum()

    feature_data["rolling_std_3"] = prior_usage.rolling(3).std()
    feature_data["rolling_std_6"] = prior_usage.rolling(6).std()
    feature_data["rolling_std_12"] = prior_usage.rolling(12).std()

    feature_data["rolling_min_12"] = prior_usage.rolling(12).min()
    feature_data["rolling_max_12"] = prior_usage.rolling(12).max()

    feature_data["trend_3"] = (
        feature_data["lag_1"] - feature_data["lag_3"]
    )

    feature_data["trend_6"] = (
        feature_data["lag_1"] - feature_data["lag_6"]
    )

    feature_data["zero_month_percentage_12"] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values: (values == 0).mean(),
            raw=True,
        )
    )

    feature_data["coefficient_variation_12"] = (
        feature_data["rolling_std_12"]
        /
        feature_data["rolling_mean_12"].replace(
            0,
            np.nan,
        )
    )

    feature_data["recent_vs_annual"] = (
        feature_data["rolling_mean_3"]
        -
        feature_data["rolling_mean_12"]
    )

    prior_commits = feature_data[COMMITS_COLUMN].shift(1)

    feature_data["commits_lag_1"] = (
        feature_data[COMMITS_COLUMN].shift(1)
    )

    feature_data["commits_lag_2"] = (
        feature_data[COMMITS_COLUMN].shift(2)
    )

    feature_data["commits_lag_3"] = (
        feature_data[COMMITS_COLUMN].shift(3)
    )

    feature_data["commits_lag_6"] = (
        feature_data[COMMITS_COLUMN].shift(6)
    )

    feature_data["commits_rolling_mean_3"] = (
        prior_commits.rolling(3).mean()
    )

    feature_data["commits_rolling_mean_6"] = (
        prior_commits.rolling(6).mean()
    )

    feature_data["commits_rolling_mean_12"] = (
        prior_commits.rolling(12).mean()
    )

    feature_data["commits_rolling_sum_3"] = (
        prior_commits.rolling(3).sum()
    )

    feature_data["commits_rolling_sum_6"] = (
        prior_commits.rolling(6).sum()
    )

    feature_data["commits_recent_vs_annual"] = (
        feature_data["commits_rolling_mean_3"]
        -
        feature_data["commits_rolling_mean_12"]
    )

    feature_columns = [
        "month",
        "year",
        "quarter",
        "time_idx",

        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",

        "rolling_mean_3",
        "rolling_mean_6",
        "rolling_mean_12",

        "rolling_median_3",
        "rolling_median_6",
        "rolling_median_12",

        "rolling_total_12",

        "rolling_std_3",
        "rolling_std_6",
        "rolling_std_12",

        "rolling_min_12",
        "rolling_max_12",

        "zero_month_percentage_12",
        "coefficient_variation_12",
        "recent_vs_annual",

        "trend_3",
        "trend_6",

        "commits_lag_1",
        "commits_lag_2",
        "commits_lag_3",
        "commits_lag_6",

        "commits_rolling_mean_3",
        "commits_rolling_mean_6",
        "commits_rolling_mean_12",

        "commits_rolling_sum_3",
        "commits_rolling_sum_6",

        "commits_recent_vs_annual",
    ]

    training_rows = (
        feature_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=feature_columns
        )
        .reset_index(drop=True)
    )

    return (
        training_rows,
        feature_columns,
    )


print(
    "Version 8 usage + commitment "
    "feature function created."
)


In [ ]:
def train_model(training_rows, feature_columns):

    # Recency weighting:
    # oldest usable training row = 1.0
    # newest usable training row = 2.0
    sample_weights = np.linspace(
        1.0,
        2.0,
        len(training_rows),
    )

    model = LGBMRegressor(
        objective="poisson",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        min_child_samples=10,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        training_rows[feature_columns],
        training_rows[TARGET_COLUMN],
        sample_weight=sample_weights,
    )

    return model


print(
    "Version 8 training function created "
    "with recency weighting."
)


In [ ]:
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
                COMMITS_COLUMN,
            ]
        ]
        .copy()
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    predictions = []

    # We do not know future commitment snapshots after June 2026.
    # Hold the latest known commitment level constant for the
    # recursive July-December forecast rather than using future data.
    latest_known_commits = float(
        forecast_history[COMMITS_COLUMN].iloc[-1]
    )

    for _ in range(forecast_months):

        next_date = (
            forecast_history[DATE_COLUMN].max()
            + pd.DateOffset(months=1)
        )

        recent_usage = forecast_history[TARGET_COLUMN]
        recent_commits = forecast_history[COMMITS_COLUMN]

        future_row = {
            "month": next_date.month,
            "year": next_date.year,
            "quarter": next_date.quarter,
            "time_idx": len(forecast_history),
        }

        # Usage lag features
        for lag in LAGS:
            future_row[f"lag_{lag}"] = (
                recent_usage.iloc[-lag]
            )

        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]

        future_row["rolling_mean_3"] = last_3.mean()
        future_row["rolling_mean_6"] = last_6.mean()
        future_row["rolling_mean_12"] = last_12.mean()

        future_row["rolling_median_3"] = last_3.median()
        future_row["rolling_median_6"] = last_6.median()
        future_row["rolling_median_12"] = last_12.median()

        future_row["rolling_total_12"] = last_12.sum()

        future_row["rolling_std_3"] = last_3.std()
        future_row["rolling_std_6"] = last_6.std()
        future_row["rolling_std_12"] = last_12.std()

        future_row["rolling_min_12"] = last_12.min()
        future_row["rolling_max_12"] = last_12.max()

        future_row["zero_month_percentage_12"] = (
            last_12.eq(0).mean()
        )

        rolling_mean_12 = future_row["rolling_mean_12"]
        rolling_std_12 = future_row["rolling_std_12"]

        if rolling_mean_12 != 0:
            future_row["coefficient_variation_12"] = (
                rolling_std_12 / rolling_mean_12
            )
        else:
            future_row["coefficient_variation_12"] = 0.0

        future_row["recent_vs_annual"] = (
            future_row["rolling_mean_3"]
            - future_row["rolling_mean_12"]
        )

        future_row["trend_3"] = (
            recent_usage.iloc[-1]
            - recent_usage.iloc[-3]
        )

        future_row["trend_6"] = (
            recent_usage.iloc[-1]
            - recent_usage.iloc[-6]
        )

        # Commitment features
        last_commits_3 = recent_commits.iloc[-3:]
        last_commits_6 = recent_commits.iloc[-6:]
        last_commits_12 = recent_commits.iloc[-12:]

        future_row["commits_lag_1"] = recent_commits.iloc[-1]
        future_row["commits_lag_2"] = recent_commits.iloc[-2]
        future_row["commits_lag_3"] = recent_commits.iloc[-3]
        future_row["commits_lag_6"] = recent_commits.iloc[-6]

        future_row["commits_rolling_mean_3"] = (
            last_commits_3.mean()
        )

        future_row["commits_rolling_mean_6"] = (
            last_commits_6.mean()
        )

        future_row["commits_rolling_mean_12"] = (
            last_commits_12.mean()
        )

        future_row["commits_rolling_sum_3"] = (
            last_commits_3.sum()
        )

        future_row["commits_rolling_sum_6"] = (
            last_commits_6.sum()
        )

        future_row["commits_recent_vs_annual"] = (
            future_row["commits_rolling_mean_3"]
            - future_row["commits_rolling_mean_12"]
        )

        future_features = pd.DataFrame(
            [future_row],
            columns=feature_columns,
        )

        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )

        predicted_usage = max(
            0.0,
            predicted_usage,
        )

        predictions.append(
            {
                PART_COLUMN: part_number,
                DATE_COLUMN: next_date,
                "Predicted Usage": predicted_usage,
            }
        )

        # Recursive forecast:
        # use predicted usage for the next month's lag/rolling features.
        # Future commitment snapshots are unknown, so carry forward
        # the latest value known at the June 2026 cutoff.
        new_history_row = pd.DataFrame(
            {
                DATE_COLUMN: [next_date],
                TARGET_COLUMN: [predicted_usage],
                COMMITS_COLUMN: [latest_known_commits],
            }
        )

        forecast_history = pd.concat(
            [
                forecast_history,
                new_history_row,
            ],
            ignore_index=True,
        )

    return pd.DataFrame(predictions)


print(
    "Version 8 recursive future forecast "
    "function created."
)


In [ ]:
all_forecasts = []
errors = []

cutoff_date = pd.Timestamp(
    TRAINING_CUTOFF
)


for part_number in sorted(
    data[PART_COLUMN].dropna().unique()
):

    print(f"\nProcessing: {part_number}")

    try:

        part_data = (
            data[
                data[PART_COLUMN] == part_number
            ]
            .copy()
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )

        # Hard cutoff: nothing dated July 2026 or later
        # can be used for training or feature creation.
        historical_data = part_data[
            part_data[DATE_COLUMN] < cutoff_date
        ].copy()

        if historical_data.empty:
            raise ValueError(
                "No historical data available before cutoff."
            )

        training_rows, feature_columns = (
            create_training_features(
                historical_data
            )
        )

        if training_rows.empty:
            raise ValueError(
                "No usable training rows after feature creation."
            )

        # Train once using only information available
        # through June 2026.
        model = train_model(
            training_rows,
            feature_columns,
        )

        # Forecast all six future months recursively.
        forecast = forecast_future_months(
            model=model,
            historical_data=historical_data,
            feature_columns=feature_columns,
            forecast_months=FORECAST_MONTHS,
            part_number=part_number,
        )

        forecast["Training Through"] = (
            historical_data[DATE_COLUMN].max()
        )

        all_forecasts.append(
            forecast
        )

        print(f"Finished: {part_number}")

    except Exception as error:

        errors.append(
            {
                PART_COLUMN: str(part_number),
                "Error": str(error),
            }
        )

        print(
            f"Skipped {part_number}: "
            f"{error}"
        )


if not all_forecasts:
    raise RuntimeError(
        "No July-December forecasts completed successfully."
    )


print(
    "\nVersion 8 July-December 2026 "
    "future forecast complete."
)


In [ ]:
results = (
    pd.concat(
        all_forecasts,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

results["Predicted Usage Rounded"] = (
    results["Predicted Usage"]
    .round()
    .astype(int)
)


display(
    results[
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Training Through",
            "Predicted Usage Rounded",
            "Predicted Usage",
        ]
    ]
)


In [ ]:
profile_data = data[
    data[DATE_COLUMN]
    < pd.Timestamp(TRAINING_CUTOFF)
].copy()


demand_profile = (
    profile_data
    .groupby(PART_COLUMN)
    .agg(
        Average_Monthly_Usage=(
            TARGET_COLUMN,
            "mean",
        ),
        Median_Monthly_Usage=(
            TARGET_COLUMN,
            "median",
        ),
        Zero_Month_Percentage=(
            TARGET_COLUMN,
            lambda x: (x == 0).mean(),
        ),
        Nonzero_Months=(
            TARGET_COLUMN,
            lambda x: (x > 0).sum(),
        ),
        Average_Commits=(
            COMMITS_COLUMN,
            "mean",
        ),
        Median_Commits=(
            COMMITS_COLUMN,
            "median",
        ),
        Maximum_Commits=(
            COMMITS_COLUMN,
            "max",
        ),
        Months_With_Commits=(
            COMMITS_COLUMN,
            lambda x: (x > 0).sum(),
        ),
        Commit_Month_Percentage=(
            COMMITS_COLUMN,
            lambda x: (x > 0).mean(),
        ),
    )
    .reset_index()
)


demand_profile["Zero_Month_Percentage"] *= 100
demand_profile["Commit_Month_Percentage"] *= 100


for col in [
    "Average_Monthly_Usage",
    "Median_Monthly_Usage",
    "Average_Commits",
    "Median_Commits",
]:
    demand_profile[col] = demand_profile[col].round(2)


for col in [
    "Zero_Month_Percentage",
    "Commit_Month_Percentage",
]:
    demand_profile[col] = demand_profile[col].round(1)


display(demand_profile)


In [ ]:
import boto3
from sagemaker_studio import Project
import io


proj = Project()
project_s3_root = proj.s3.root


s3_parts = (
    project_s3_root
    .replace("s3://", "")
    .split("/", 1)
)


bucket = s3_parts[0]
prefix = (
    s3_parts[1]
    if len(s3_parts) > 1
    else ""
)


s3 = boto3.client("s3")


results_to_upload = results[
    [
        PART_COLUMN,
        DATE_COLUMN,
        "Training Through",
        "Predicted Usage",
        "Predicted Usage Rounded",
    ]
].copy()


csv_buffer = io.StringIO()


results_to_upload.to_csv(
    csv_buffer,
    index=False,
)


s3_key = (
    f"{prefix}/results/"
    "version_8_july_december_2026_future_forecast.csv"
)


s3.put_object(
    Bucket=bucket,
    Key=s3_key,
    Body=csv_buffer.getvalue().encode(
        "utf-8"
    ),
    ContentType="text/csv",
)


s3_path = (
    f"s3://{bucket}/{s3_key}"
)


print(
    "Successfully uploaded Version 8 "
    "July-December 2026 future forecast to S3!"
)

print(f"S3 path: {s3_path}")
print(f"Rows uploaded: {len(results_to_upload)}")
